# Pump It Up final model and candidate submission

Freeze the equal-weight Random Forest and histogram-boosting vote selected in `03-model-comparison.ipynb`. It led the five development folds at **81.37% mean accuracy**. No settings or ensemble components are changed after observing the local test.

This notebook performs the two final operations only:

1. fit on development data and inspect the reserved local test once;
2. refit the unchanged components on all labelled `original` rows and classify the unlabelled `competition` rows.

The fitting, probability alignment and submission validation live in `src/final_model.py` so the notebook remains a readable record of the decision.

In [1]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != 'notebooks' or NOTEBOOK_DIR.parent.name != 'stage-1-pump-it-up':
    raise RuntimeError(
        'Run this notebook from the stage-1-pump-it-up/notebooks directory.'
    )

STAGE_DIR = NOTEBOOK_DIR.parent
DATA_DIR = STAGE_DIR / 'data'
SRC_DIR = STAGE_DIR / 'src'
SUBMISSION_DIR = (
    STAGE_DIR / 'submissions' / '2026-08-21-forest-boosting'
)
SUBMISSION_PATH = (
    SUBMISSION_DIR / '01-random-forest-histogram-boosting.csv'
)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_partitioning import partition_modelling_data
from final_model import (
    SELECTED_WORKFLOW_NAME,
    evaluate_selected_workflow_on_local_test,
    fit_selected_workflow_for_competition,
    write_validated_submission,
)
from modelling_data import prepare_modelling_data

## Reconstruct the frozen data roles

`original` is the complete labelled dataset before local partitioning. `competition` is the separate unlabelled dataset supplied for DrivenData scoring. Identifiers remain metadata rather than predictors.

In [2]:
raw_original = pd.read_csv(DATA_DIR / 'TrainingSetValues.csv')
labels_original = pd.read_csv(DATA_DIR / 'TrainingSetLabels.csv')
raw_competition = pd.read_csv(DATA_DIR / 'TestSetValues.csv')
submission_template = pd.read_csv(DATA_DIR / 'SubmissionFormat.csv')

modelling_data = prepare_modelling_data(
    raw_original,
    labels_original,
    raw_competition,
)
partitioned_data = partition_modelling_data(modelling_data)

## One-time local-test evaluation

Fit the already-selected workflow on all 47,520 development rows and score the 11,880 local-test rows. This result confirms the frozen selection; it is not used to choose different components or settings.

In [3]:
local_test_evaluation = evaluate_selected_workflow_on_local_test(
    partitioned_data
)
as_percentages = lambda values: values.map(lambda value: f'{value:.2%}')

print(f'Selected workflow: {SELECTED_WORKFLOW_NAME}')
print('\nLocal-test metrics')
print(as_percentages(local_test_evaluation.metrics).to_string())
print('\nConfusion counts')
print(local_test_evaluation.confusion_counts.to_string())
print('\nRecall-normalised confusion matrix')
print(
    as_percentages(local_test_evaluation.confusion_recall).to_string()
)
print('\nComponent fit and prediction seconds')
print(local_test_evaluation.component_seconds.round(1).to_string())

Selected workflow: equal-weight Random Forest + histogram boosting

Local-test metrics
accuracy                           80.82%
recall: functional                 89.48%
recall: functional needs repair    32.10%
recall: non functional             77.81%

Confusion counts
predicted                functional  functional needs repair  non functional
actual                                                                      
functional                     5773                      155             524
functional needs repair         455                      277             131
non functional                  959                       54            3552

Recall-normalised confusion matrix
predicted               functional functional needs repair non functional
actual                                                                   
functional                  89.48%                   2.40%          8.12%
functional needs repair     52.72%                  32.10%         15.18%
non functi

### Local-test interpretation

The selected workflow reaches **80.82% local-test accuracy**, 0.55 percentage points below its 81.37% development-fold mean. This modest reduction supports the selection rather than indicating a material validation collapse.

Repair recall is **32.10%**, 2.18 points below the development-fold mean. The class remains the model's clearest weakness and a priority for the next iteration, but no post-test change is made to today's workflow.

## Refit on all labels and classify the competition rows

The local test has completed its evaluation role. Recreate both component models, fit them on all 59,400 labelled `original` rows, average their aligned class probabilities and validate the resulting 14,850-row submission against the supplied template.

In [4]:
competition_prediction = fit_selected_workflow_for_competition(
    modelling_data,
    submission_template,
)
written_path = write_validated_submission(
    competition_prediction,
    SUBMISSION_PATH,
)

print(f'Validated submission: {written_path}')
print(f'Rows: {len(competition_prediction.submission):,}')
print('\nFull-data fit and prediction seconds')
print(competition_prediction.component_seconds.round(1).to_string())
print('\nCompetition prediction shares')
print(as_percentages(competition_prediction.class_shares).to_string())

Validated submission: C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\stage-1-pump-it-up\submissions\2026-08-21-forest-boosting\01-random-forest-histogram-boosting.csv
Rows: 14,850

Full-data fit and prediction seconds
Random Forest         53.3
histogram boosting    39.2

Competition prediction shares
status_group
functional                 60.59%
functional needs repair     3.82%
non functional             35.58%


## Outcome

The candidate CSV is structurally ready for DrivenData upload. Its public score is unknown until submission and must be recorded separately rather than inferred from local evidence.

The next modelling investigation should examine the meaning and possible ordinal or hierarchical structure of the three target classes using development-only out-of-fold evidence. Grouped geographic sensitivity also remains an outstanding robustness check.